# 01 · Explore the historical MBTA bus arrival/departure data
One monthly file, opened before anything is loaded into Postgres. Six questions from Task 2;
write your answer under each one.

In [20]:
import glob
import pandas as pd

path = sorted(glob.glob("../data/raw/*.csv"))[0]
df = pd.read_csv(path, dtype=str)
print(path, f"{len(df):,} rows")
df.head()

../data/raw/MBTA-Bus-Arrival-Departure-Times_2026-04.csv 2,459,855 rows


,service_date,route_id,direction_id,half_trip_id,stop_id,time_point_id,time_point_order,point_type,standard_type,scheduled,actual,scheduled_headway,headway
0,2026-04-01,01,Inbound,NaN,64,nubn,1,Endpoint,Headway,1900-01-01T11:35:00Z,1900-01-01T11:45:36Z,NaN,NaN
1,2026-04-16,01,Inbound,NaN,64,nubn,1,Endpoint,Headway,1900-01-01T17:18:00Z,1900-01-01T17:24:49Z,NaN,NaN
2,2026-04-25,01,Inbound,NaN,64,nubn,1,Endpoint,Headway,1900-01-01T21:41:00Z,1900-01-01T21:44:53Z,NaN,NaN
3,2026-04-24,215,Inbound,NaN,3359,emlsq,1,Endpoint,Schedule,1900-01-01T19:50:00Z,1900-01-01T19:52:38Z,1620,NaN
4,2026-04-05,23,Inbound,NaN,64000,nubn,1,Endpoint,Headway,1900-01-02T00:37:00Z,1900-01-02T00:25:58Z,NaN,NaN


## 1. How are the times encoded?

In [21]:
print(df["scheduled"].head(3).tolist())
print("date part of scheduled:", df["scheduled"].str[:10].value_counts().to_dict())

['1900-01-01T11:35:00Z', '1900-01-01T17:18:00Z', '1900-01-01T21:41:00Z']
date part of scheduled: {'1900-01-01': 2348081, '1900-01-02': 111774}


**Answer:** The times are Boston clock times on a fake 1900 date, so I take the date from `service_date`, and 1900-01-02 just means the bus ran after midnight.

## 2. How many rows have no `actual`?

In [22]:
print(f"{df['actual'].isna().sum():,} rows ({df['actual'].isna().mean():.1%})")

141,081 rows (5.7%)


**Answer:** 141,081 rows (5.7%) have no `actual` (no recorded arrival time). These rows are kept, but they received no late label.

## 3. What does the delay look like?

In [23]:
scheduled = pd.to_datetime(df["scheduled"].str[:19])
actual = pd.to_datetime(df["actual"].str[:19])
delay = (actual - scheduled).dt.total_seconds() / 60
print(delay.describe(percentiles=[.01, .05, .5, .95, .99]).round(2))
print("outside [-30, +120]:", ((delay < -30) | (delay > 120)).sum())
print("late rate (> 3 min):", round((delay[(delay >= -30) & (delay <= 120)] > 3).mean(), 4))


count    2318774.00
mean           3.19
std            5.54
min          -54.98
1%            -9.45
5%            -4.03
50%            2.35
95%           13.08
99%           21.27
max          325.27
dtype: float64
outside [-30, +120]: 231
late rate (> 3 min): 0.4334


**Answer:**
- The median delay is about 2.4 minutes, so a typical bus runs a little late
- Most delays fall between about -4 and +13 minutes (5%-95% range)
- There's a long tail on the late side, up to over 5 hours, which is a data error
- Only 231 rows are outside -30 to +120, and the late rate is about 43%

## 4. Is the key (service date, route, direction, stop, scheduled) unique?

In [24]:
key = ["service_date", "route_id", "direction_id", "stop_id", "scheduled"]
dup = df.duplicated(key, keep=False)
print(f"{dup.sum():,} rows share a key with another row")
df[dup].sort_values(key).head(6)

3,382 rows share a key with another row


,service_date,route_id,direction_id,half_trip_id,stop_id,time_point_id,time_point_order,point_type,standard_type,scheduled,actual,scheduled_headway,headway
1475,2026-04-01,09,Inbound,70391419,150,brway,4,Midpoint,Schedule,1900-01-01T06:50:00Z,NaN,720,NaN
23119,2026-04-01,09,Inbound,70394552,150,brway,4,Midpoint,Headway,1900-01-01T06:50:00Z,1900-01-01T06:54:15Z,NaN,NaN
476,2026-04-01,10,Outbound,70391262,20,perks,6,Midpoint,Schedule,1900-01-01T14:09:00Z,1900-01-01T14:07:20Z,2400,NaN
23192,2026-04-01,10,Outbound,70394562,20,perks,5,Midpoint,Schedule,1900-01-01T14:09:00Z,1900-01-01T14:18:21Z,2400,NaN
36327,2026-04-01,101,Inbound,70396204,29001,sull,6,Endpoint,Schedule,1900-01-01T15:14:00Z,1900-01-01T15:17:14Z,1740,NaN
39399,2026-04-01,101,Inbound,70396678,29001,sull,7,Endpoint,Schedule,1900-01-01T15:14:00Z,1900-01-01T15:31:21Z,1740,NaN


In [25]:
shared = df[dup]
trips_per_key = shared.groupby(key)["half_trip_id"].transform("nunique")
same_trip = shared[trips_per_key == 1].sort_values(key)

print(f"{same_trip.groupby(key).ngroups} keys are shared by the SAME half_trip_id")
same_trip[key + ["half_trip_id", "time_point_order", "point_type", "actual"]].head(6)

63 keys are shared by the SAME half_trip_id


,service_date,route_id,direction_id,stop_id,scheduled,half_trip_id,time_point_order,point_type,actual
40887,2026-04-01,101,Inbound,91771,1900-01-01T14:50:00Z,70396923,1,Startpoint,NaN
40888,2026-04-01,101,Inbound,91771,1900-01-01T14:50:00Z,70396923,2,Startpoint,NaN
39825,2026-04-01,101,Inbound,91771,1900-01-01T15:00:00Z,70396751,1,Startpoint,NaN
39826,2026-04-01,101,Inbound,91771,1900-01-01T15:00:00Z,70396751,2,Startpoint,NaN
36847,2026-04-01,134,Inbound,91771,1900-01-01T15:00:00Z,70396292,1,Startpoint,1900-01-01T14:54:37Z
36848,2026-04-01,134,Inbound,91771,1900-01-01T15:00:00Z,70396292,2,Startpoint,NaN


**Answer:**
- No, the key isn't unique
- 3,382 rows (about 0.14%) share a key. Most are two different trips (different `half_trip_id`) at the same stop in the same minute, but in 63 cases it's the same trip listed twice at one stop (`time_point_order` 1 and 2)
- The table keeps one row per key, preferring the one with an `actual`

## 5. Which columns are half empty, and why?

In [26]:
print(df.isna().mean().round(3).sort_values(ascending=False))
pd.crosstab(df["standard_type"], df["scheduled_headway"].isna())

scheduled_headway    0.545
headway              0.544
actual               0.057
service_date         0.000
route_id             0.000
direction_id         0.000
half_trip_id         0.000
stop_id              0.000
time_point_id        0.000
time_point_order     0.000
point_type           0.000
standard_type        0.000
scheduled            0.000
dtype: float64


scheduled_headway,False,True
standard_type,,
Headway,0,1275094
Schedule,1119764,64997


**Answer:**
- Only `scheduled_headway` and `headway` are about half empty (54%)
- The crosstab shows `scheduled_headway` is only filled when `standard_type` is Schedule. It's always empty for Headway rows
- `headway` is the measured gap between buses, and it's empty in about the same rows

## 6. What do the route and stop ids look like?

In [27]:
routes = sorted(df["route_id"].unique())
print(len(routes), "routes:", routes[:12], "...", routes[-8:])
print("stop ids:", df["stop_id"].drop_duplicates().head(10).tolist())

152 routes: ['01', '04', '07', '08', '09', '10', '100', '101', '104', '105', '106', '108'] ... ['99', 'CT2', 'CT3', 'SL1', 'SL2', 'SL3', 'SL4', 'SL5']
stop ids: ['64', '3359', '64000', '32003', '38493', '17861', '18511', '150', '899', '18178']


**Answer:**
- There are 152 routes. Most are numbers with zero-padding (01, 04); others have letters (CT2, CT3, SL1-SL5)
- Stop ids are numbers of different lengths (64, 3359, 64000)
- Both must stay text, not numbers, or 01 would be 1 and stop matching